# 3.0 Face AutoKeras Baseline

Purpose:
- Load the cleaned face train/validation/test split
- Balance the training data using augmentation
- Train an AutoKeras image classifier
- Evaluate on the untouched test set
- Save predictions and evaluation results

Important:
Only the training set is augmented. Validation and test data remain unchanged.

In [1]:
import numpy as np
import pandas as pd
import autokeras as ak

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    FACE_SPLIT_MANIFEST_PATH,
    FACE_IMAGE_SIZE,
    FACE_AUTOKERAS_DIR,
    FACE_AUTOKERAS_MODEL_PATH,
    FACE_BASELINE_RESULTS_PATH,
    FACE_PREDICTIONS_PATH,
    FACE_CONFUSION_MATRIX_PATH,
)
from rural_stroke_assist.preprocessing.face_dataset import (
    load_face_split_arrays,
    compute_class_weights,
    INVERSE_LABEL_MAPPING,
)
from rural_stroke_assist.modeling.image_augmentation import (
    apply_augmentation_to_training_data,
)
from rural_stroke_assist.modeling.evaluation import (
    evaluate_binary_classifier,
    create_classification_report_dict,
    save_confusion_matrix_plot,
    save_markdown_results,
)

c:\Users\Asus\Downloads\Uni_work\Grad-Project\autokeras_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
face_split_df = pd.read_csv(FACE_SPLIT_MANIFEST_PATH)

face_split_df.head()

,path,file_name,class_label,extension,readable,width,height,file_hash,is_clean,split
0,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0022.jpg,NonStroke,.jpg,True,299,169,a3e6d954dd74d17e89c497d4967c93366d623b4b5b694f...,True,train
1,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0024.jpg,NonStroke,.jpg,True,379,379,b2a1a613cdb309d96a998ed46adea5fe182aa661a40c88...,True,test
2,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0025.jpg,NonStroke,.jpg,True,275,183,7038e96c4cafff98b47b9df614b9cd51473476e4d99699...,True,train
3,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0027.jpg,NonStroke,.jpg,True,163,222,b9a1bd4960d3afa939c7f908c1d821c468ebd247e92f50...,True,train
4,C:\Users\Asus\Downloads\Uni_work\Grad-Project\...,img_0030.jpg,NonStroke,.jpg,True,342,382,bac0af9d5eef5722d3b989d1e29677f6d4b479bda29748...,True,test


In [3]:
face_split_df.groupby(["split", "class_label"]).size()

split  class_label
test   NonStroke      208
       Stroke         110
train  NonStroke      970
       Stroke         510
val    NonStroke      208
       Stroke         109
dtype: int64

In [4]:
X_train, y_train, train_df = load_face_split_arrays(
    face_split_df,
    split="train",
    image_size=FACE_IMAGE_SIZE,
)

X_val, y_val, val_df = load_face_split_arrays(
    face_split_df,
    split="val",
    image_size=FACE_IMAGE_SIZE,
)

X_test, y_test, test_df = load_face_split_arrays(
    face_split_df,
    split="test",
    image_size=FACE_IMAGE_SIZE,
)

X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((1480, 224, 224, 3),
 (1480,),
 (317, 224, 224, 3),
 (317,),
 (318, 224, 224, 3),
 (318,))

In [5]:
np.unique(y_train, return_counts=True)

(array([0, 1]), array([970, 510], dtype=int64))

In [6]:
X_train_aug, y_train_aug = apply_augmentation_to_training_data(
    X_train=X_train,
    y_train=y_train,
    target_minority_ratio=0.50,
    minority_class=1,
    seed=42,
)

np.unique(y_train_aug, return_counts=True)

(array([0, 1]), array([970, 970], dtype=int64))

In [7]:
X_train_aug = X_train_aug / 255.0
X_val = X_val / 255.0
X_test = X_test / 255.0

In [8]:
class_weights = compute_class_weights(y_train_aug)
class_weights

{0: 1.0, 1: 1.0}

In [9]:
clf = ak.ImageClassifier(
    overwrite=True,
    max_trials=5,
    project_name="face_stroke_binary_baseline",
    directory=str(FACE_AUTOKERAS_DIR),
)

clf.fit(
    x=X_train_aug,
    y=y_train_aug,
    validation_data=(X_val, y_val),
    epochs=10,
)

Trial 2 Complete [01h 41m 07s]
val_loss: 0.9851958155632019

Best val_loss So Far: 0.7018154859542847
Total elapsed time: 02h 05m 09s

Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
efficient         |vanilla           |image_block_1/block_type
True              |True              |image_block_1/normalize
True              |False             |image_block_1/augment
True              |None              |image_block_1/image_augmentation_1/horizontal_flip
False             |None              |image_block_1/image_augmentation_1/vertical_flip
0                 |None              |image_block_1/image_augmentation_1/contrast_factor
0                 |None              |image_block_1/image_augmentation_1/rotation_factor
0.1               |None              |image_block_1/image_augmentation_1/translation_factor
0                 |None              |image_block_1/image_augmentation_1/zoom_factor
True              |None              |image_block_1/efficient_net_bloc

: 